# Week 6 Exercise: Category-Specific Fine-Tuning

**Goal:** Fine-tune GPT-4.1-nano to specialize in pricing products by category (Electronics, Toys, Automotive, etc.) and compare specialist models against a general model.

**Experiment:**
1. Load dataset and group items by category
2. Fine-tune a **general model** on mixed categories (baseline)
3. Fine-tune **category-specific models** for 2–3 categories with enough data
4. Evaluate specialists vs general model per category with visualizations

In [ ]:
# Imports
import os
from pathlib import Path
import json
from collections import defaultdict
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items import Item
from pricer.evaluator import evaluate
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Environment
load_dotenv(override=True)
hf_token = os.environ["HF_TOKEN"]
login(hf_token, add_to_git_credential=True)

# Fine-tuning requires OpenAI API. Jobs appear at https://platform.openai.com/finetune
# OpenRouter does NOT support fine-tuning (inference-only).
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Optional: OpenRouter client for base-model inference only (not fine-tuned models).
# Use this for zero-shot comparisons. Fine-tuned models (ft:...) are OpenAI-only.
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
openrouter_client = (
    OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)
    if OPENROUTER_API_KEY
    else None
)

# Load data - use items_lite (has LLM-generated summaries) for fine-tuning
username = "ed-donner"
dataset = f"{username}/items_lite"
train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test")

# Group by category
def by_category(items):
    d = defaultdict(list)
    for item in items:
        d[item.category].append(item)
    return dict(d)

train_by_cat = by_category(train)
val_by_cat = by_category(val)
test_by_cat = by_category(test)

# Check counts per category
for cat in sorted(train_by_cat.keys()):
    print(f"{cat}: train={len(train_by_cat[cat])}, val={len(val_by_cat[cat])}, test={len(test_by_cat[cat])}")


## 1. Visual: Items per Category

In [ ]:
# Bar chart of training items per category
categories = sorted(train_by_cat.keys())
counts = [len(train_by_cat[c]) for c in categories]

plt.figure(figsize=(12, 5))
plt.bar(categories, counts, color="steelblue")
plt.title("Training Items per Category")
plt.xlabel("Category")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
for i, v in enumerate(counts):
    plt.text(i, v, str(v), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 2. Select Categories & Prepare Fine-Tuning Data

We pick categories with at least 50 train and 10 val examples. Use 100 train / 25 val per model.

In [ ]:
MIN_TRAIN = 50
MIN_VAL = 10
TRAIN_SIZE = 100
VAL_SIZE = 25

# Select categories with enough data
selected_categories = [
    cat for cat in sorted(train_by_cat.keys())
    if len(train_by_cat[cat]) >= MIN_TRAIN and len(val_by_cat[cat]) >= MIN_VAL
]
print(f"Selected categories: {selected_categories}")

In [ ]:
# JSONL helpers (same format as day5)
def messages_for(item, category=None):
    prompt = f"Estimate the price of this product. Respond with the price, no explanation"
    if category:
        prompt = f"Estimate the price of this {category.replace('_', ' ')} product. Respond with the price, no explanation"
    return [
        {"role": "user", "content": f"{prompt}\n\n{item.summary}"},
        {"role": "assistant", "content": f"${item.price:.2f}"},
    ]

def make_jsonl(items, category=None):
    result = ""
    for item in items:
        msgs = messages_for(item, category)
        result += '{"messages": ' + json.dumps(msgs) + "}\n"
    return result.strip()

def write_jsonl(items, filename, category=None):
    Path(filename).parent.mkdir(parents=True, exist_ok=True)
    with open(filename, "w") as f:
        f.write(make_jsonl(items, category))

## 3. Create JSONL Files

- **General model:** 100 mixed items from all categories
- **Category specialists:** 100 items each from Electronics, Toys_and_Games, etc.

In [ ]:
import random
random.seed(42)

# General: sample 100 from across all categories
general_train = []
per_cat = TRAIN_SIZE // len(selected_categories)
for cat in selected_categories:
    general_train.extend(random.sample(train_by_cat[cat], min(per_cat, len(train_by_cat[cat]))))
random.shuffle(general_train)
general_train = general_train[:TRAIN_SIZE]
general_val = val[:VAL_SIZE]

write_jsonl(general_train, "jsonl/general_train.jsonl")
write_jsonl(general_val, "jsonl/general_val.jsonl")
print(f"General: {len(general_train)} train, {len(general_val)} val")

In [ ]:
# Category specialists - use first 2-3 categories for cost control
SPECIALIST_CATEGORIES = selected_categories[:2]
print(f"Fine-tuning specialists for: {SPECIALIST_CATEGORIES}")

for cat in SPECIALIST_CATEGORIES:
    ct = train_by_cat[cat][:TRAIN_SIZE]
    cv = val_by_cat[cat][:VAL_SIZE]
    write_jsonl(ct, f"jsonl/{cat}_train.jsonl", category=cat)
    write_jsonl(cv, f"jsonl/{cat}_val.jsonl", category=cat)
    print(f"  {cat}: {len(ct)} train, {len(cv)} val")

## 4. Upload & Start Fine-Tuning Jobs



In [ ]:
BASE_MODEL = "gpt-4.1-nano-2025-04-14"
HYPERPARAMS = {"n_epochs": 1, "batch_size": 1}

# Upload general
with open("jsonl/general_train.jsonl", "rb") as f:
    general_train_file = client.files.create(file=f, purpose="fine-tune")
with open("jsonl/general_val.jsonl", "rb") as f:
    general_val_file = client.files.create(file=f, purpose="fine-tune")

# Start general fine-tuning
general_job = client.fine_tuning.jobs.create(
    training_file=general_train_file.id,
    validation_file=general_val_file.id,
    model=BASE_MODEL,
    seed=42,
    hyperparameters=HYPERPARAMS,
    suffix="pricer-general",
)
print(f"General job: {general_job.id}")

In [ ]:
# Upload & start category specialist jobs
specialist_jobs = {}
for cat in SPECIALIST_CATEGORIES:
    with open(f"jsonl/{cat}_train.jsonl", "rb") as f:
        tf = client.files.create(file=f, purpose="fine-tune")
    with open(f"jsonl/{cat}_val.jsonl", "rb") as f:
        vf = client.files.create(file=f, purpose="fine-tune")
    job = client.fine_tuning.jobs.create(
        training_file=tf.id,
        validation_file=vf.id,
        model=BASE_MODEL,
        seed=42,
        hyperparameters=HYPERPARAMS,
        suffix=f"pricer-{cat.lower()[:20]}",
    )
    specialist_jobs[cat] = job
    print(f"{cat}: {job.id}")

## 5. Wait for Jobs to Complete

In [ ]:
import time

def poll_until_done(job_id):
    while True:
        j = client.fine_tuning.jobs.retrieve(job_id)
        status = j.status
        if status == "succeeded":
            return j.fine_tuned_model
        if status == "failed":
            raise RuntimeError(f"Job {job_id} failed")
        print(f"  {job_id}: {status}")
        time.sleep(30)

# Get general model
general_model = poll_until_done(general_job.id)
print(f"General model: {general_model}")

# Get specialist models
specialist_models = {}
for cat, job in specialist_jobs.items():
    specialist_models[cat] = poll_until_done(job.id)
    print(f"{cat}: {specialist_models[cat]}")

## 6. Evaluation: Specialist vs General

- **General model:** used for all items
- **Specialist model:** used when the item's category matches; otherwise fallback to general

In [ ]:
def test_messages_for(item):
    return [{"role": "user", "content": f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"}]

def predict(item, model_name):
    r = client.chat.completions.create(
        model=model_name,
        messages=test_messages_for(item),
        max_tokens=7,
    )
    return r.choices[0].message.content

def general_pricer(item):
    return predict(item, general_model)

def specialist_router_pricer(item):
    """Use specialist if available, else general."""
    model = specialist_models.get(item.category, general_model)
    return predict(item, model)

In [ ]:
import re
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor

def post_process(value):
    if isinstance(value, str):
        value = value.replace("$", "").replace(",", "")
        m = re.search(r"[-+]?\d*\.\d+|\d+", value)
        return float(m.group()) if m else 0
    return float(value)

def mae_on_items(predictor, items, workers=5):
    errors = []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        for guess, item in zip(tqdm(ex.map(predictor, items), total=len(items), desc="Predicting"), items):
            g = post_process(guess)
            errors.append(abs(g - item.price))
    return sum(errors) / len(errors) if errors else 0

In [ ]:
# Per-category MAE: General vs Specialist
# Use smaller eval size per category for speed (e.g. 50)
EVAL_SIZE = 50
results = []

for cat in SPECIALIST_CATEGORIES:
    test_items = test_by_cat.get(cat, [])[:EVAL_SIZE]
    if not test_items:
        continue
    mae_gen = mae_on_items(general_pricer, test_items)
    specialist_model = specialist_models.get(cat)
    mae_spec = mae_on_items(lambda i: predict(i, specialist_model), test_items) if specialist_model else None
    results.append({
        "category": cat,
        "n": len(test_items),
        "MAE (general)": mae_gen,
        "MAE (specialist)": mae_spec,
        "improvement": (mae_gen - mae_spec) if mae_spec else 0,
    })

results_df = pd.DataFrame(results)
results_df

## 7. Visual: General vs Specialist MAE by Category

In [ ]:
# Bar chart comparing General vs Specialist MAE per category
df = results_df
x = range(len(df))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - w/2 for i in x], df["MAE (general)"], w, label="General", color="steelblue")
ax.bar([i + w/2 for i in x], df["MAE (specialist)"], w, label="Specialist", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(df["category"], rotation=45, ha="right")
ax.set_ylabel("Mean Absolute Error ($)")
ax.set_title("Category-Specific Fine-Tuning: General vs Specialist MAE")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Full Test Set Evaluation (Optional)

Run `evaluate()` on the full test set with the specialist router.

In [ ]:
# Evaluate specialist router on 200 test items (uses API calls)
evaluate(specialist_router_pricer, test, size=200)

## Report Summary

- **General model:** Trained on mixed categories; baseline for comparison.
- **Specialist models:** Trained on a single category each; expected to perform better on that category.
- **Findings:** Check the bar chart. Positive improvement = specialist beats general for that category.